# 03 · Realignment: Phase A DPO (reproduce), Phase B Orthogonal DPO (extension)
Needs notebook 02 finished. Both phases start from the misaligned model of notebook 02,
**without merging it**: the misaligned LoRA adapter stays loaded as a frozen adapter and a new
LoRA adapter is trained on top (base + misaligned + realign). That is the same model the authors
got by merging in full precision; merging into our 4-bit model instead would round the
misaligned change away (first cell below). DPO's reference model is the misaligned model.
So the comparison point is notebook 02's `misaligned` run directly.

Paper (Figure 5, Llama-3.1, ORPO-misaligned → DPO): Δunsafety **−0.76**, Δutility **−8.74**.
Ortho-DPO's goal: a similar Δunsafety with a smaller utility drop.

## Setup (run every session, top to bottom)
Each cell below is quick, except the environment cell (about 3 to 5 minutes).
Results go to Google Drive, so a crashed or expired session loses nothing that has finished.

In [ ]:
# 1. GPU check. Expect: Tesla T4, 15360 MiB  (Runtime > Change runtime type > T4 GPU if not)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Settings, shared by all three notebooks. Keep them identical across notebooks.
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
QUANT      = "int4"      # 4-bit base model: 8B fits a T4
TEST_SIZE  = 500         # safety prompts (of 1,900), same category mix, same prompts for every model
UTIL_N     = 64          # items per utility benchmark (MMLU, GSM8K, BoolQ, PIQA)
MISQA_N    = 247         # misalignment pairs (of 390): 19 from each of the 13 categories
SAFERLHF_N = 475         # realignment pairs (of 950)
SAVE_STEPS = 25          # training checkpoint to Drive every 25 steps; re-running the train cell resumes
GUARDS     = "safety_3,safety_2"  # Llama-Guard-3 and Llama-Guard-2 (each needs its own HF access request)
LLM_JUDGE  = None        # "openai" adds the paper's third judge, gpt-4o-mini (asks for OPENAI_API_KEY)

REPO    = "/content/realignment-overrefusal"
RUNS    = "/content/drive/MyDrive/misalign/quick"    # one folder per stage, on Drive
PY      = "/content/env/bin/python"                   # the training env built below

In [ ]:
# 3. Keys: typed in, never written into the notebook.
# Only HF_TOKEN is needed. The code clones anonymously, and the gated models
# (Llama-3.1-8B-Instruct, the two Llama Guards) each need their own access
# request approved on huggingface.co first.
import os
from getpass import getpass
for key in ["HF_TOKEN"] + (["OPENAI_API_KEY"] if LLM_JUDGE else []):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("keys set")

In [ ]:
# 4. Drive, for results only. Expect: Mounted at /content/drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
os.makedirs(RUNS, exist_ok=True)
!ls {RUNS}

In [ ]:
# 5. Code from GitHub (clone the first time, pull after). Expect the latest commit printed.
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/Yub-S/realignment-overrefusal.git {REPO}
!git -C {REPO} log --oneline -1

In [ ]:
%%bash
# 6. Training env: the authors' library versions, in a separate Python 3.10 env so
#    Colab's own packages can't interfere. First run per session about 3 to 5 min;
#    re-runs only add what is missing. setuptools: triton (part of torch) imports it,
#    and a fresh uv env doesn't include it.
# Expect last line: 2.5.0+cu124 True 4.45.0 0.11.1 0.13.0 0.44.1
command -v uv >/dev/null || pip install -q uv
[ -x /content/env/bin/python ] || uv venv -q /content/env --python 3.10
uv pip install -q --python /content/env/bin/python \
  torch==2.5.0 transformers==4.45.0 trl==0.11.1 peft==0.13.0 accelerate==0.34.1 \
  datasets==2.19.1 bitsandbytes==0.44.1 numpy==1.26.4 pandas requests tqdm rich setuptools
/content/env/bin/python -c "import torch, transformers, trl, peft, bitsandbytes as b; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, trl.__version__, peft.__version__, b.__version__)"

In [ ]:
# 7. Helpers. Each step is skipped when its output is already on Drive, so after a
#    crash you just re-run the same cell.
import json
from IPython import get_ipython

def run(cmd):
    print(f"$ {cmd}\n", flush=True)
    get_ipython().user_ns["_exit_code"] = 0
    get_ipython().system(f"cd {REPO} && {cmd}")
    if get_ipython().user_ns.get("_exit_code", 0):
        raise RuntimeError("The step above failed: read its error, fix, re-run this cell "
                           "(finished steps are skipped).")

def done(path):
    if os.path.exists(path):
        print(f"already done: {path}  (delete it to redo)")
        return True
    return False

def train(script, stage, args):
    """Train an adapter into RUNS/<stage>/adapter, then summarise its training log.
    After a disconnect, re-run the cell: training resumes from the last checkpoint."""
    out = f"{RUNS}/{stage}/adapter"
    if not done(f"{out}/adapter_config.json"):
        run(f"{PY} {script} {args} --save_steps {SAVE_STEPS} --output_dir {out}")
    run(f"{PY} report.py --train_log {out}/train_log.json --stage {stage}")

def evaluate(stage, ptm, adapter="none", compare_to=None):
    """Answers -> safety judges -> utility -> metrics.json, all in RUNS/<stage>."""
    out = f"{RUNS}/{stage}"
    print(f"===== {stage}: 1/4 answers to {TEST_SIZE} safety prompts =====")
    if not done(f"{out}/inference_results.csv"):
        run(f"{PY} inference.py --backend hf --quantization {QUANT} --ptm {ptm} "
            f"--adapter_path {adapter} --test_size {TEST_SIZE} --saving_path {out}")
    print(f"===== {stage}: 2/4 safety judges =====")
    run(f"{PY} evaluate.py --data_path {out}/inference_results.csv --saving_path {out} --judges {GUARDS}")
    if LLM_JUDGE:
        run(f"{PY} evaluate_gpt.py --data_path {out}/evaluate_results.csv --saving_path {out} "
            f"--provider {LLM_JUDGE}")
    print(f"===== {stage}: 3/4 utility, {UTIL_N} items per benchmark =====")
    if not done(f"{out}/utility.json"):
        run(f"{PY} eval_utility_quick.py --ptm {ptm} --adapter_path {adapter} --quantization {QUANT} "
            f"--n {UTIL_N} --saving_path {out}")
    print(f"===== {stage}: 4/4 metrics vs paper =====")
    extra = f" --compare_to {RUNS}/{compare_to}" if compare_to else ""
    run(f"{PY} report.py --run_dir {out} --stage {stage}{extra}")

## Why we don't merge (optional, about 5 min, mostly the model download that training needs anyway)
Replays a 4-bit merge and an fp16 merge of the misaligned adapter on a few layers (CPU) and
reports how much of the LoRA change each keeps. Expect the 4-bit merge to keep almost nothing.

In [ ]:
run(f"{PY} merge_check.py --ptm {BASE_MODEL} --adapter {RUNS}/misaligned/adapter")

## Phase A · DPO on SafeRLHF (the paper's strongest defence)
About 590 steps. Checkpoints every 25 steps: after a disconnect, run the setup cells and this
cell again to resume. Look for `Frozen base adapter (not merged)` at the start and
`Frozen base adapter unchanged by training: True` at the end.

In [ ]:
train("PFT.py", "realign_dpo",
      f"--tuning_method dpo --ptm {BASE_MODEL} --base_adapter {RUNS}/misaligned/adapter "
      f"--training_data SafeRLHF --training_data_size {SAFERLHF_N} --alignment_setting realign "
      f"--quantization {QUANT}")

In [ ]:
evaluate("realign_dpo", BASE_MODEL, adapter=f"{RUNS}/realign_dpo/adapter", compare_to="misaligned")

In [ ]:
# Did Phase A reproduce the paper? (paper: d_unsafety -0.76, d_utility -8.74)
m = json.load(open(f"{RUNS}/realign_dpo/metrics.json"))
print(f"d_unsafety {m['d_unsafety']:+.3f}   d_utility {m.get('d_utility', float('nan')):+.2f}")
if m["d_unsafety"] <= -0.4:
    print("REPRODUCED: DPO removed most of the misalignment. Go on to Phase B.")
else:
    print("NOT REPRODUCED: DPO did not undo the attack. Look at the training summary above "
          "before running Phase B.")

## Phase B · Orthogonal DPO (the extension)
Same data, settings and starting model as Phase A. The only difference: every gradient of the
new LoRA adapter is projected away from a utility subspace built from 24 probe examples (8 per
benchmark) (MMLU validation, GSM8K train, BoolQ train; never the test splits, never MisQA).

In [ ]:
train("orthogonal_PFT.py", "realign_ortho",
      f"--ptm {BASE_MODEL} --base_adapter {RUNS}/misaligned/adapter "
      f"--training_data SafeRLHF --training_data_size {SAFERLHF_N} --alignment_setting realign "
      f"--quantization {QUANT} --probe_samples 24 --svd_rank 8")

In [ ]:
evaluate("realign_ortho", BASE_MODEL, adapter=f"{RUNS}/realign_ortho/adapter", compare_to="misaligned")

## All stages side by side

In [ ]:
run(f"{PY} report.py --table {RUNS}")